# German Electricity Load Forecasting

## Objective

This notebook prepares the German electricity dataset for short-term
electricity demand forecasting.

We will:

1. Load and preserve the original dataset
2. Validate timestamps and data structure
3. Explore electricity demand and missing periods
4. Identify suspicious or corrupted observations
5. Clean the dataset
6. Create continuous time-series segments
7. Engineer lag and calendar features
8. Prepare the final modelling dataset
9. Perform a chronological train/test split
10. Train and evaluate forecasting models

### Important data-handling rule

`df_raw` will always remain an untouched copy of the original CSV.

All cleaning operations will be performed on `df`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

# 1. Load the Original Dataset

We first load the CSV exactly as provided.

No cleaning, filtering, interpolation, or feature engineering is performed
at this stage.

In [ ]:
file_path = r"C:\Users\Grace\OneDrive\Documents\Colleye TY\5th Semester\GridAI\data\power_consumption_germany.csv"

df_raw = pd.read_csv(file_path)

print("Original dataset loaded successfully.")
print("Shape:", df_raw.shape)

## 2. Verify the Original Dataset

Before performing any cleaning or transformation, we inspect the original
dataset to understand its structure, data types, columns, timestamps, and
potential data-quality issues.

The original `df_raw` dataframe will remain unchanged.

In [ ]:
print("Dataset shape:", df_raw.shape)

print("\nNumber of rows:", len(df_raw))
print("Number of columns:", len(df_raw.columns))

print("\nColumn names:")
print(df_raw.columns.tolist())

In [ ]:
print("First 5 rows:")
display(df_raw.head())

print("\nLast 5 rows:")
display(df_raw.tail())

In [ ]:
print("Data types:")
display(df_raw.dtypes)

In [ ]:
df_raw.info()

# 3. Timestamp and Data Continuity Exploration

The dataset contains a timestamp column stored under `Unnamed: 0`.

In this section, we will:

1. Convert the timestamp into a proper datetime representation.
2. Check the chronological range of the dataset.
3. Check whether timestamps are duplicated.
4. Check the expected 15-minute sampling interval.
5. Identify genuine gaps in the time series.
6. Determine whether the apparent missing period near the end of the dataset is actually missing data or simply rows with unavailable measurements.

No observations will be deleted in this section.

In [ ]:
df = df_raw.copy()

print("Working dataframe created.")
print("Shape:", df.shape)

In [ ]:
df = df.rename(columns={"Unnamed: 0": "timestamp"})

print("Timestamp column renamed successfully.")
print("First columns:")
print(df.columns[:10].tolist())

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

print("Timestamp conversion completed.")
print("Timestamp dtype:", df["timestamp"].dtype)

In [ ]:
print("Earliest timestamp:", df["timestamp"].min())
print("Latest timestamp:", df["timestamp"].max())
print("Total observations:", len(df))

In [ ]:
duplicate_timestamps = df["timestamp"].duplicated().sum()

print("Duplicate timestamps:", duplicate_timestamps)

In [ ]:
is_sorted = df["timestamp"].is_monotonic_increasing

print("Timestamps already sorted:", is_sorted)

In [ ]:
df = df.sort_values("timestamp").reset_index(drop=True)

print("Data sorted chronologically.")
print("First timestamp:", df["timestamp"].iloc[0])
print("Last timestamp:", df["timestamp"].iloc[-1])

In [ ]:
df["time_gap"] = df["timestamp"].diff()

print("Most common timestamp gaps:")
display(df["time_gap"].value_counts().head(10))

In [ ]:
large_gaps = df.loc[
    df["time_gap"] > pd.Timedelta(hours=1),
    ["timestamp", "time_gap"]
]

display(large_gaps)

In [ ]:
gap_start = pd.Timestamp("2018-09-30 21:45:00", tz="UTC")
gap_end = pd.Timestamp("2018-12-31 23:00:00", tz="UTC")

print("Rows around the 2018 gap:")

display(
    df[
        df["timestamp"].between(
            gap_start - pd.Timedelta(hours=1),
            gap_end + pd.Timedelta(hours=1)
        )
    ][["timestamp", "load_Actual Load"]]
)

In [ ]:
expected_timestamps = pd.date_range(
    start=df["timestamp"].min(),
    end=df["timestamp"].max(),
    freq="15min",
    tz="UTC"
)

actual_timestamps = pd.DatetimeIndex(df["timestamp"])

missing_timestamps = expected_timestamps.difference(actual_timestamps)

print("Expected timestamps:", len(expected_timestamps))
print("Actual timestamps:", len(actual_timestamps))
print("Missing timestamps:", len(missing_timestamps))

In [ ]:
if len(missing_timestamps) > 0:
    missing_series = pd.Series(missing_timestamps)

    missing_groups = (
        missing_series.diff() != pd.Timedelta(minutes=15)
    ).cumsum()

    missing_periods = missing_series.groupby(missing_groups).agg(
        start="min",
        end="max",
        count="size"
    )

    missing_periods["duration"] = (
        missing_periods["end"]
        - missing_periods["start"]
        + pd.Timedelta(minutes=15)
    )

    display(missing_periods)
else:
    print("No missing timestamps found.")

In [ ]:
print("First 10 ORIGINAL timestamp values:")
display(df_raw["Unnamed: 0"].head(10))

print("\nLast 10 ORIGINAL timestamp values:")
display(df_raw["Unnamed: 0"].tail(10))

# 4. Missing Value Analysis

The timestamp structure and chronological ordering have been verified.

We now examine missing values in the measurement variables. This step only
quantifies missing data; no values or rows will be modified yet.

The timestamp column is excluded because it represents the time index rather
than a measurement.

In [ ]:
# Identify measurement columns
measurement_columns = [
    col for col in df.columns
    if col not in ["timestamp", "time_gap"]
]

# Calculate missing values
missing_summary = pd.DataFrame({
    "missing_count": df[measurement_columns].isna().sum(),
    "missing_percentage": (
        df[measurement_columns].isna().mean() * 100
    ).round(2)
})

# Sort by percentage of missing values
missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_summary)

## Missing Value Analysis Complete

The table above shows the number and percentage of missing observations
for each measurement variable.

No missing values have been filled, interpolated, or deleted at this stage.

The missing-data patterns will be used to determine the appropriate
cleaning strategy for each type of variable.

In [ ]:
# Keep only the original measurement variables
measurement_columns = [
    col for col in df.columns
    if col not in ["timestamp", "timestamp_local", "time_gap", "local_time_gap"]
]

# Recalculate missing-value summary
missing_summary = pd.DataFrame({
    "missing_count": df[measurement_columns].isna().sum(),
    "missing_percentage": (
        df[measurement_columns].isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_summary)

# 5. Missingness Pattern Classification

The missing-value analysis shows that missingness is not uniform across
the dataset.

Some variables have very high levels of missing data, while others have
only a small number of missing observations.

Instead of applying one imputation method to every variable, the variables
will first be classified according to their missing-data percentage.

This prevents inappropriate interpolation of variables that contain large
historical periods where the measurement was not available.

In [ ]:
# Classify variables according to their percentage of missing values

def classify_missingness(pct):
    if pct == 0:
        return "No missing values"
    elif pct <= 1:
        return "Very low (<=1%)"
    elif pct <= 5:
        return "Low (1-5%)"
    elif pct <= 50:
        return "Moderate (5-50%)"
    else:
        return "High (>50%)"

missing_summary["missing_category"] = (
    missing_summary["missing_percentage"]
    .apply(classify_missingness)
)

display(missing_summary)

In [ ]:
# Summary of missingness categories

missing_category_counts = (
    missing_summary["missing_category"]
    .value_counts()
)

display(missing_category_counts.to_frame("number_of_variables"))

# Missingness Classification Complete

The variables have now been classified according to the proportion of
missing observations.

No rows or columns have been deleted, and no missing values have been
filled.

Variables with substantial missingness will be examined before deciding
whether they should be removed or retained. Variables with small gaps may
be suitable for time-series interpolation, subject to their role in the
forecasting model.

In [ ]:
# Display variables in each missingness category

for category in [
    "No missing values",
    "Very low (<=1%)",
    "Low (1-5%)",
    "Moderate (5-50%)",
    "High (>50%)"
]:
    print("\n" + "=" * 70)
    print(category)
    print("=" * 70)
    
    cols = missing_summary[
        missing_summary["missing_category"] == category
    ].index.tolist()
    
    for col in cols:
        print(col)

# 6. Missing Data Strategy

The missing-data analysis shows that the dataset contains several distinct
missingness patterns.

Variables with very low missingness are generally suitable for time-series
interpolation because only a small number of observations are unavailable.

Variables with moderate missingness require greater care because large gaps
may make interpolation unreliable.

Variables with more than 50% missing observations will not be blindly
interpolated. Their usefulness will first be evaluated based on their
coverage and role in the forecasting problem.

The target variable, `load_Actual Load`, has only a very small proportion of
missing observations and will be treated separately from predictor variables.

No observations are deleted at this stage.

In [ ]:
# Check missing values in the target variable

target_column = "load_Actual Load"

print("Target variable:", target_column)
print("Missing values:", df[target_column].isna().sum())
print(
    "Missing percentage:",
    round(df[target_column].isna().mean() * 100, 2),
    "%"
)

# 7. Data Cleaning

Based on the missing-value analysis, the dataset contains variables with
very different levels of data availability.

The following cleaning strategy is adopted:

1. Variables with more than 5% missing values will be removed because
   large missing periods cannot be reliably reconstructed through
   interpolation.

2. Variables with 5% or less missing values will be retained.

3. Small missing gaps in retained predictor variables will be filled using
   time-based interpolation.

4. Rows where the target variable (`load_Actual Load`) is missing will
   not be artificially generated. These rows will be removed from the
   final modelling dataset.

5. The original dataframe (`df_raw`) will remain unchanged throughout
   the cleaning process.

This approach avoids creating artificial long-term data while preserving
variables with sufficiently reliable coverage.

In [ ]:
# Define the target
target_column = "load_Actual Load"

# Keep variables with at most 5% missing values
columns_to_keep = missing_summary[
    missing_summary["missing_percentage"] <= 5
].index.tolist()

# Columns with more than 5% missing values will be removed
columns_to_remove = missing_summary[
    missing_summary["missing_percentage"] > 5
].index.tolist()

print("Number of measurement columns:", len(measurement_columns))
print("Columns retained:", len(columns_to_keep))
print("Columns removed:", len(columns_to_remove))

print("\nColumns removed because of >5% missingness:")
for col in columns_to_remove:
    print("-", col)

In [ ]:
# Create cleaned dataframe
df_clean = df.copy()

# Remove variables with excessive missingness
df_clean = df_clean.drop(columns=columns_to_remove)

print("Columns removed successfully.")
print("New shape:", df_clean.shape)

# 8. Interpolation of Small Missing Gaps

The retained predictor variables contain only small amounts of missing data.

Because the dataset is a time series with observations every 15 minutes,
short missing gaps can be estimated using time-based interpolation.

Interpolation is performed only for retained measurement variables and
does not affect the original dataset.

The target variable is handled separately to avoid creating artificial
target observations for model training. 

In [ ]:
# Check remaining missing values

remaining_missing = df_clean.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

print("Remaining missing values:")

if len(remaining_missing) == 0:
    print("None. All retained variables are complete.")
else:
    display(
        remaining_missing
        .sort_values(ascending=False)
        .to_frame("missing_count")
    )

In [ ]:
# Remove predictors that still contain missing values
# after limited interpolation

remaining_predictor_missing = [
    col for col in df_clean.columns
    if col != target_column and df_clean[col].isna().any()
]

print("Predictor columns removed:")
for col in remaining_predictor_missing:
    print(f"- {col}")

df_clean = df_clean.drop(columns=remaining_predictor_missing)

# Remove rows where the target is missing
target_missing_before = df_clean[target_column].isna().sum()

df_clean = df_clean.dropna(subset=[target_column])

print(f"\nRows removed because target was missing: {target_missing_before}")
print(f"Final shape after cleaning: {df_clean.shape}")

In [ ]:
# Final validation of cleaned dataset

print("Final df_clean shape:", df_clean.shape)
print("Index:", df_clean.index.name)
print("Target:", target_column)

# Check remaining missing values
remaining_missing = df_clean.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

print("\nRemaining missing values:")
if remaining_missing.empty:
    print("None — df_clean is ready for EDA.")
else:
    print(remaining_missing.sort_values(ascending=False))

# Check target
print("\nTarget missing values:", df_clean[target_column].isna().sum())

# Check chronological order
print("Chronologically sorted:", df_clean.index.is_monotonic_increasing)

# Check duplicate timestamps
print("Duplicate timestamps:", df_clean.index.duplicated().sum())

In [ ]:
print("Shape:", df_clean.shape)

In [ ]:
# EDA 1 — Basic target statistics

target = df_clean[target_column]

print("Target variable:", target_column)
print("\nBasic statistics:")
display(target.describe())

print("\nAdditional statistics:")
print("Minimum load:", target.min())
print("Maximum load:", target.max())
print("Mean load:", target.mean())
print("Median load:", target.median())
print("Standard deviation:", target.std())

In [ ]:
# Rebuild df_clean from the working dataframe using our finalized cleaning rules

target_column = "load_Actual Load"

# Start fresh from the working dataframe
df_clean = df.copy()

# Keep only variables with <= 5% missing values
columns_to_keep = missing_summary[
    missing_summary["missing_percentage"] <= 5
].index.tolist()

# Make sure the target is retained
if target_column not in columns_to_keep:
    columns_to_keep.append(target_column)

# Keep timestamp plus selected measurement variables
df_clean = df_clean[
    ["timestamp"] + columns_to_keep
].copy()

# Set timestamp as index
df_clean = df_clean.set_index("timestamp").sort_index()

# Interpolate only short internal gaps in predictors
predictor_columns = [
    col for col in df_clean.columns
    if col != target_column
]

for col in predictor_columns:
    if pd.api.types.is_numeric_dtype(df_clean[col]):
        df_clean[col] = df_clean[col].interpolate(
            method="linear",
            limit=4,
            limit_area="inside"
        )

# Remove any predictor that still has missing values
remaining_missing_predictors = [
    col for col in predictor_columns
    if df_clean[col].isna().any()
]

df_clean = df_clean.drop(columns=remaining_missing_predictors)

# Remove rows where the target is missing
df_clean = df_clean.dropna(subset=[target_column])

print("Cleaned dataframe rebuilt successfully.")
print("Final shape:", df_clean.shape)
print("Remaining missing values:", df_clean.isna().sum().sum())
print("Target missing values:", df_clean[target_column].isna().sum())
print("Chronologically sorted:", df_clean.index.is_monotonic_increasing)
print("Duplicate timestamps:", df_clean.index.duplicated().sum())

In [ ]:
# Rebuild df_clean directly from the original raw dataset
# This avoids relying on the stale missing_summary variable.

target_column = "load_Actual Load"

# Original measurement columns from the untouched raw dataset
measurement_columns = [
    col for col in df_raw.columns
    if col != "Unnamed: 0"
]

# Calculate missing percentages directly
missing_pct = (
    df[measurement_columns]
    .isna()
    .mean()
    .mul(100)
)

# Keep variables with <= 5% missing values
columns_to_keep = missing_pct[
    missing_pct <= 5
].index.tolist()

# Always retain the target
if target_column not in columns_to_keep:
    columns_to_keep.append(target_column)

# Build clean dataframe
df_clean = df[["timestamp"] + columns_to_keep].copy()

# Set timestamp as index
df_clean = df_clean.set_index("timestamp").sort_index()

# Separate predictors from target
predictor_columns = [
    col for col in df_clean.columns
    if col != target_column
]

# Fill only short internal gaps
for col in predictor_columns:
    if pd.api.types.is_numeric_dtype(df_clean[col]):
        df_clean[col] = df_clean[col].interpolate(
            method="linear",
            limit=4,
            limit_area="inside"
        )

# Remove predictors that still contain missing values
remaining_predictor_missing = [
    col for col in predictor_columns
    if df_clean[col].isna().any()
]

df_clean = df_clean.drop(columns=remaining_predictor_missing)

# Remove rows where the target is missing
df_clean = df_clean.dropna(subset=[target_column])

print("Cleaned dataframe rebuilt.")
print("Final shape:", df_clean.shape)
print("Remaining missing values:", df_clean.isna().sum().sum())
print("Target missing values:", df_clean[target_column].isna().sum())
print("Chronologically sorted:", df_clean.index.is_monotonic_increasing)
print("Duplicate timestamps:", df_clean.index.duplicated().sum())

print("\nRemaining columns:")
print(df_clean.columns.tolist())

In [ ]:
# Final rebuild of modelling dataset

target_column = "load_Actual Load"

# Start from the original working dataframe
df_clean = df.copy()

# Original measurement columns
measurement_columns = [
    col for col in df_clean.columns
    if col != "timestamp"
]

# Calculate missing percentage directly
missing_pct = (
    df_clean[measurement_columns]
    .isna()
    .mean() * 100
)

# Keep variables with <= 5% missing values
columns_to_keep = missing_pct[
    missing_pct <= 5
].index.tolist()

# Always keep the target
if target_column not in columns_to_keep:
    columns_to_keep.append(target_column)

# Select columns
df_clean = df_clean[
    ["timestamp"] + columns_to_keep
].copy()

# Set timestamp as index
df_clean = df_clean.set_index("timestamp").sort_index()

# Predictor columns
predictor_columns = [
    col for col in df_clean.columns
    if col != target_column
]

# Interpolate short internal gaps only
for col in predictor_columns:
    if pd.api.types.is_numeric_dtype(df_clean[col]):
        df_clean[col] = df_clean[col].interpolate(
            method="linear",
            limit=4,
            limit_area="inside"
        )

# Remove rows where target is missing
df_clean = df_clean.dropna(subset=[target_column])

print("FINAL CLEANING COMPLETE")
print("=" * 40)
print("Shape:", df_clean.shape)
print("Target:", target_column)
print("Remaining missing values:", df_clean.isna().sum().sum())
print("Target missing values:", df_clean[target_column].isna().sum())
print("Sorted:", df_clean.index.is_monotonic_increasing)
print("Duplicate timestamps:", df_clean.index.duplicated().sum())
print("\nColumns retained:", len(df_clean.columns))
print(df_clean.columns.tolist())

In [ ]:
# FINAL CLEAN DATASET REBUILD
# Uses df_raw directly to avoid any stale notebook variables

target_column = "load_Actual Load"

# Start from untouched raw data
df_clean = df_raw.copy()

# Rename timestamp column
df_clean = df_clean.rename(columns={"Unnamed: 0": "timestamp"})

# Convert timestamps to UTC
df_clean["timestamp"] = pd.to_datetime(
    df_clean["timestamp"],
    utc=True
)

# Calculate missing percentage directly from the raw data
measurement_columns = [
    col for col in df_clean.columns
    if col != "timestamp"
]

missing_pct = (
    df_clean[measurement_columns]
    .isna()
    .mean() * 100
)

# Keep variables with <= 5% missing values
columns_to_keep = missing_pct[
    missing_pct <= 5
].index.tolist()

# Keep timestamp + selected variables
df_clean = df_clean[
    ["timestamp"] + columns_to_keep
].copy()

# Sort and set timestamp as index
df_clean = (
    df_clean
    .sort_values("timestamp")
    .set_index("timestamp")
)

# Identify predictors
predictor_columns = [
    col for col in df_clean.columns
    if col != target_column
]

# Interpolate only short internal gaps
for col in predictor_columns:
    if pd.api.types.is_numeric_dtype(df_clean[col]):
        df_clean[col] = df_clean[col].interpolate(
            method="linear",
            limit=4,
            limit_area="inside"
        )

# Remove rows where target is missing
target_missing = df_clean[target_column].isna().sum()
df_clean = df_clean.dropna(subset=[target_column])

# Report final dataset
remaining_missing = df_clean.isna().sum()
remaining_missing = remaining_missing[
    remaining_missing > 0
].sort_values(ascending=False)

print("FINAL CLEAN DATASET")
print("=" * 50)
print("Shape:", df_clean.shape)
print("Number of variables:", len(df_clean.columns))
print("Target:", target_column)
print("Rows removed due to missing target:", target_missing)

print("\nRemaining predictor missing values:")
if remaining_missing.empty:
    print("None")
else:
    display(
        remaining_missing.to_frame("missing_count")
    )

print("\nValidation:")
print("Target missing:", df_clean[target_column].isna().sum())
print("Sorted:", df_clean.index.is_monotonic_increasing)
print("Duplicate timestamps:", df_clean.index.duplicated().sum())

In [ ]:
# EDA 1 — Target distribution and extreme values

target = df_clean[target_column]

print("Target:", target_column)
print("\nDescriptive statistics:")
display(target.describe())

# Extreme-value investigation
q1 = target.quantile(0.25)
q3 = target.quantile(0.75)
iqr = q3 - q1

upper_iqr = q3 + 1.5 * iqr

print("\nOutlier thresholds:")
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Upper IQR threshold:", upper_iqr)

extreme_values = df_clean[
    df_clean[target_column] > upper_iqr
][[target_column]]

print("\nNumber of IQR outliers:", len(extreme_values))

print("\nLargest 10 load values:")
display(
    df_clean[[target_column]]
    .nlargest(10, target_column)
)

In [ ]:
# Investigate the extreme load anomaly

anomaly_time = pd.Timestamp("2026-06-25 16:30:00", tz="UTC")

anomaly_window = df_clean.loc[
    anomaly_time - pd.Timedelta(hours=1):
    anomaly_time + pd.Timedelta(hours=1),
    [target_column]
]

print("Load values around the suspicious observation:")
display(anomaly_window)

print("\nAnomalous value:", df_clean.loc[anomaly_time, target_column])

# Compare with neighboring observations
before = df_clean.loc[
    anomaly_time - pd.Timedelta(minutes=15),
    target_column
]

after = df_clean.loc[
    anomaly_time + pd.Timedelta(minutes=15),
    target_column
]

print("\nPrevious 15-min value:", before)
print("Next 15-min value:", after)

print(
    "\nRatio of anomaly to neighboring average:",
    df_clean.loc[anomaly_time, target_column] /
    ((before + after) / 2)
)

In [ ]:
# Correct the confirmed isolated load anomaly

anomaly_time = pd.Timestamp("2026-06-25 16:30:00", tz="UTC")

previous_value = df_clean.loc[
    anomaly_time - pd.Timedelta(minutes=15),
    target_column
]

next_value = df_clean.loc[
    anomaly_time + pd.Timedelta(minutes=15),
    target_column
]

# Linear interpolation between the neighboring observations
corrected_value = (previous_value + next_value) / 2

# Store the original value for documentation
original_anomaly_value = df_clean.loc[anomaly_time, target_column]

# Correct only this single observation
df_clean.loc[anomaly_time, target_column] = corrected_value

print("Anomaly corrected successfully.")
print("Timestamp:", anomaly_time)
print("Original value:", original_anomaly_value)
print("Corrected value:", corrected_value)
print("Previous value:", previous_value)
print("Next value:", next_value)

In [ ]:
# EDA 2 — Electricity load over the full study period

import matplotlib.pyplot as plt

plt.figure(figsize=(16, 5))

plt.plot(
    df_clean.index,
    df_clean[target_column],
    linewidth=0.5
)

plt.title("German Electricity Load Over Time")
plt.xlabel("Time")
plt.ylabel("Actual Load")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# EDA 3 — Average electricity load by hour of day

hourly_profile = (
    df_clean[target_column]
    .groupby(df_clean.index.hour)
    .mean()
)

plt.figure(figsize=(12, 5))

plt.plot(
    hourly_profile.index,
    hourly_profile.values,
    marker="o"
)

plt.title("Average Electricity Load by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Average Actual Load")
plt.xticks(range(24))
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# EDA 4 — Average load by day of week

weekday_profile = (
    df_clean[target_column]
    .groupby(df_clean.index.dayofweek)
    .mean()
)

days = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

plt.figure(figsize=(11, 5))

plt.plot(
    days,
    weekday_profile.values,
    marker="o"
)

plt.title("Average Electricity Load by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Actual Load")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# EDA 5 — Average electricity load by day and hour
# Calendar features are calculated using German local time

local_time = df_clean.index.tz_convert("Europe/Berlin")

weekly_hourly = (
    df_clean[target_column]
    .groupby([local_time.dayofweek, local_time.hour])
    .mean()
    .unstack()
)

days = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

weekly_hourly.index = days

plt.figure(figsize=(14, 6))

plt.imshow(
    weekly_hourly,
    aspect="auto",
    interpolation="nearest"
)

plt.colorbar(label="Average Actual Load")

plt.title("Average Electricity Load by Day and Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")

plt.xticks(range(24), range(24))
plt.yticks(range(7), days)

plt.tight_layout()
plt.show()

In [ ]:
# EDA 6 — Average electricity load by month
# Use German local time for calendar-based analysis

local_time = df_clean.index.tz_convert("Europe/Berlin")

monthly_profile = (
    df_clean[target_column]
    .groupby(local_time.month)
    .mean()
)

months = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

plt.figure(figsize=(12, 5))

plt.plot(
    months,
    monthly_profile.values,
    marker="o"
)

plt.title("Average Electricity Load by Month")
plt.xlabel("Month")
plt.ylabel("Average Actual Load")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# EDA 7 — Distribution of actual electricity load

plt.figure(figsize=(12, 5))

plt.hist(
    df_clean[target_column],
    bins=60
)

plt.title("Distribution of Actual Electricity Load")
plt.xlabel("Actual Load")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# EDA 8 — Correlation of variables with actual load

correlation_with_target = (
    df_clean
    .corr(numeric_only=True)[target_column]
    .drop(target_column)
    .sort_values(ascending=False)
)

print("Variables most positively correlated with actual load:")
display(
    correlation_with_target.head(10).to_frame("correlation")
)

print("\nVariables most negatively correlated with actual load:")
display(
    correlation_with_target.tail(10).to_frame("correlation")
)

In [ ]:
# EDA 9 — Strongest predictor vs actual load

strongest_predictor = correlation_with_target.index[0]

print("Strongest positively correlated predictor:")
print(strongest_predictor)
print("Correlation:", correlation_with_target.iloc[0])

plt.figure(figsize=(10, 6))

plt.scatter(
    df_clean[strongest_predictor],
    df_clean[target_column],
    alpha=0.15,
    s=8
)

plt.title(
    f"Actual Load vs {strongest_predictor}"
)
plt.xlabel(strongest_predictor)
plt.ylabel("Actual Load")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# EDA 10 — Top predictors by absolute correlation

top_correlations = (
    correlation_with_target
    .abs()
    .sort_values(ascending=False)
    .head(10)
)

top_correlation_table = (
    correlation_with_target
    .loc[top_correlations.index]
    .to_frame("correlation")
)

display(top_correlation_table)

In [ ]:
# Feature Engineering 1 — Calendar features

local_time = df_clean.index.tz_convert("Europe/Berlin")

df_clean["hour"] = local_time.hour
df_clean["minute"] = local_time.minute
df_clean["day_of_week"] = local_time.dayofweek
df_clean["month"] = local_time.month
df_clean["day_of_year"] = local_time.dayofyear
df_clean["is_weekend"] = (local_time.dayofweek >= 5).astype(int)

print("Calendar features created successfully.")
print("\nNew features:")
print([
    "hour",
    "minute",
    "day_of_week",
    "month",
    "day_of_year",
    "is_weekend"
])

print("\nUpdated shape:", df_clean.shape)

In [ ]:
# Feature Engineering 2 — Gap-aware lag features

# Make sure the data is in chronological order
df_clean = df_clean.sort_index()

# Create lag features using the actual timestamps
# This prevents large missing timestamp gaps from being treated as normal intervals.

for lag, feature_name in [
    (pd.Timedelta(minutes=15), "lag_15min"),
    (pd.Timedelta(hours=1), "lag_1hour"),
    (pd.Timedelta(days=1), "lag_1day"),
    (pd.Timedelta(days=7), "lag_1week")
]:
    
    previous_timestamp = df_clean.index - lag
    
    lag_series = pd.Series(
        df_clean[target_column].values,
        index=df_clean.index
    )
    
    df_clean[feature_name] = lag_series.reindex(previous_timestamp).values


print("Lag features created successfully.")

print("\nNew lag features:")
print([
    "lag_15min",
    "lag_1hour",
    "lag_1day",
    "lag_1week"
])

print("\nUpdated shape:", df_clean.shape)

print("\nMissing values in lag features:")
display(
    df_clean[
        ["lag_15min", "lag_1hour", "lag_1day", "lag_1week"]
    ].isna().sum().to_frame("missing_count")
)

In [ ]:
# Feature Engineering 3 — Rolling load features

# Rolling statistics based only on previous load observations
# Shift by 1 so the current target is never included.

df_clean["rolling_mean_1hour"] = (
    df_clean[target_column]
    .shift(1)
    .rolling(window=4)
    .mean()
)

df_clean["rolling_mean_1day"] = (
    df_clean[target_column]
    .shift(1)
    .rolling(window=96)
    .mean()
)

df_clean["rolling_std_1day"] = (
    df_clean[target_column]
    .shift(1)
    .rolling(window=96)
    .std()
)

print("Rolling features created successfully.")

print("\nNew rolling features:")
print([
    "rolling_mean_1hour",
    "rolling_mean_1day",
    "rolling_std_1day"
])

print("\nUpdated shape:", df_clean.shape)

print("\nMissing values in rolling features:")
display(
    df_clean[
        [
            "rolling_mean_1hour",
            "rolling_mean_1day",
            "rolling_std_1day"
        ]
    ].isna().sum().to_frame("missing_count")
)

In [ ]:
# Feature Engineering 4 — Prepare final modelling dataset

# Keep df_clean unchanged
df_model = df_clean.copy()

# Features that require historical observations
lag_features = [
    "lag_15min",
    "lag_1hour",
    "lag_1day",
    "lag_1week"
]

rolling_features = [
    "rolling_mean_1hour",
    "rolling_mean_1day",
    "rolling_std_1day"
]

history_features = lag_features + rolling_features

# Remove rows where historical features are unavailable
rows_before = len(df_model)

df_model = df_model.dropna(subset=history_features)

rows_removed = rows_before - len(df_model)

print("Final modelling dataset prepared.")
print("\nRows before:", rows_before)
print("Rows removed due to unavailable historical features:", rows_removed)
print("Rows after:", len(df_model))
print("Number of variables:", df_model.shape[1])

print("\nRemaining missing values:", df_model.isna().sum().sum())

In [ ]:
# Feature Engineering 5 — Remove rows with remaining missing predictors

# Keep the target separate
predictor_columns = [
    col for col in df_model.columns
    if col != target_column
]

rows_before = len(df_model)

# Remove rows where any predictor is still missing
df_model = df_model.dropna(subset=predictor_columns)

rows_removed = rows_before - len(df_model)

print("Final modelling dataset completed.")
print()
print("Rows before:", rows_before)
print("Rows removed due to missing predictors:", rows_removed)
print("Rows after:", len(df_model))
print("Variables:", df_model.shape[1])

print()
print("Remaining missing values:", df_model.isna().sum().sum())
print("Target missing values:", df_model[target_column].isna().sum())
print("Chronologically sorted:", df_model.index.is_monotonic_increasing)
print("Duplicate timestamps:", df_model.index.duplicated().sum())

## 9. Chronological Train-Validation-Test Split

Since this is a time-series forecasting problem, the data is split chronologically rather than randomly.

The dataset is divided into:
- **70% Training set** — used to train the ML model
- **15% Validation set** — used for model tuning and comparison
- **15% Test set** — used for final evaluation

This ensures that future observations are never used to train the model on past observations, preventing data leakage.

In [ ]:
# Model Preparation 1 — Chronological train/validation/test split

n = len(df_model)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train_data = df_model.iloc[:train_end].copy()
validation_data = df_model.iloc[train_end:validation_end].copy()
test_data = df_model.iloc[validation_end:].copy()

print("Chronological data split completed.")
print()
print("Training set:")
print("Rows:", len(train_data))
print("Start:", train_data.index.min())
print("End:", train_data.index.max())

print("\nValidation set:")
print("Rows:", len(validation_data))
print("Start:", validation_data.index.min())
print("End:", validation_data.index.max())

print("\nTest set:")
print("Rows:", len(test_data))
print("Start:", test_data.index.min())
print("End:", test_data.index.max())

## 10. Prepare Features and Target

The target variable is the actual electricity load. All remaining variables are used as input features for the machine learning model.

The target is separated from the predictors for the training, validation, and test sets.

In [ ]:
# Model Preparation 2 — Separate features and target

X_train = train_data.drop(columns=[target_column])
y_train = train_data[target_column]

X_validation = validation_data.drop(columns=[target_column])
y_validation = validation_data[target_column]

X_test = test_data.drop(columns=[target_column])
y_test = test_data[target_column]

print("Features and target separated successfully.")
print()

print("Training:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nValidation:")
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("\nTest:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nNumber of input features:", X_train.shape[1])

## 11. Baseline Forecast

Before training the machine learning model, a simple persistence baseline is established.

The baseline predicts electricity load using the load observed 15 minutes earlier. This provides a reference point for evaluating whether the machine learning model provides a meaningful improvement over a simple historical-load prediction.

In [ ]:
# Model 1 — 15-minute persistence baseline

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Use the previous 15-minute load as the prediction
baseline_validation_pred = validation_data["lag_15min"]
baseline_test_pred = test_data["lag_15min"]

# Validation metrics
baseline_val_mae = mean_absolute_error(
    y_validation,
    baseline_validation_pred
)

baseline_val_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        baseline_validation_pred
    )
)

# Test metrics
baseline_test_mae = mean_absolute_error(
    y_test,
    baseline_test_pred
)

baseline_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_test_pred
    )
)

print("15-Minute Persistence Baseline")
print("=" * 40)

print("\nValidation:")
print("MAE :", baseline_val_mae)
print("RMSE:", baseline_val_rmse)

print("\nTest:")
print("MAE :", baseline_test_mae)
print("RMSE:", baseline_test_rmse)

## 12. XGBoost Time-Series Forecasting Model

An XGBoost regression model is used to forecast actual electricity load.

The model uses historical load information, lag features, rolling statistics, calendar features, and the retained electricity-system variables.

The model is trained only on historical observations and evaluated on later validation and test periods.

In [ ]:
# Install XGBoost and create the forecasting model

%pip install xgboost -q

from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("XGBoost installed and model created successfully.")

## 12.1 Train the XGBoost Model

The XGBoost regression model is trained using the training portion of the time-series dataset.

The validation and test sets are kept separate to evaluate how well the model generalizes to future observations.

In [ ]:
# Train the XGBoost forecasting model

print("Training XGBoost model...")
print("Training samples:", len(X_train))
print("Number of features:", X_train.shape[1])

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_validation, y_validation)],
    verbose=False
)

print("\nXGBoost model trained successfully.")

## 12.2 XGBoost Model Evaluation

The trained XGBoost model is evaluated on the validation and test datasets using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).

These results are compared with the 15-minute persistence baseline to determine whether the machine learning model provides a meaningful improvement.

In [ ]:
# Evaluate XGBoost on validation and test sets

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Validation predictions
y_val_pred = xgb_model.predict(X_validation)

# Test predictions
y_test_pred = xgb_model.predict(X_test)

# Calculate metrics
xgb_val_mae = mean_absolute_error(y_validation, y_val_pred)
xgb_val_rmse = np.sqrt(mean_squared_error(y_validation, y_val_pred))

xgb_test_mae = mean_absolute_error(y_test, y_test_pred)
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print("XGBoost Model Performance")
print("=" * 40)

print("\nValidation:")
print("MAE :", xgb_val_mae)
print("RMSE:", xgb_val_rmse)

print("\nTest:")
print("MAE :", xgb_test_mae)
print("RMSE:", xgb_test_rmse)

print("\nComparison with 15-Minute Persistence Baseline:")
print("Validation MAE improvement:",
      baseline_val_mae - xgb_val_mae)

print("Test MAE improvement:",
      baseline_test_mae - xgb_test_mae)

## 12.3 Actual vs Predicted Load

The XGBoost predictions are compared with the actual electricity load over a selected period.

This visualization helps assess how closely the machine learning model follows the real load pattern and where prediction errors occur.

In [ ]:
# Visualize actual vs predicted load on the test set

import matplotlib.pyplot as plt

# Select the first 7 days of the test set
n_points = 7 * 24 * 4

plt.figure(figsize=(15, 6))

plt.plot(
    y_test.index[:n_points],
    y_test.iloc[:n_points],
    label="Actual Load"
)

plt.plot(
    y_test.index[:n_points],
    y_test_pred[:n_points],
    label="XGBoost Prediction"
)

plt.title("Actual vs Predicted Electricity Load - XGBoost")
plt.xlabel("Time")
plt.ylabel("Electricity Load")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12.4 Prediction Error Analysis

Prediction errors are analyzed to identify periods where the XGBoost model performs well or struggles.

The residual is calculated as the difference between the actual electricity load and the predicted load.

In [ ]:
# Calculate prediction errors

import numpy as np
import matplotlib.pyplot as plt

errors = y_test.values - y_test_pred

print("Prediction Error Statistics")
print("===========================")
print("Mean Error:", np.mean(errors))
print("Mean Absolute Error:", np.mean(np.abs(errors)))
print("Minimum Error:", np.min(errors))
print("Maximum Error:", np.max(errors))
print("Standard Deviation of Error:", np.std(errors))

# Plot prediction errors for the first 7 days of the test set
n_points = 7 * 24 * 4

plt.figure(figsize=(15, 5))

plt.plot(
    y_test.index[:n_points],
    errors[:n_points]
)

plt.axhline(0, linestyle="--")

plt.title("XGBoost Prediction Errors")
plt.xlabel("Time")
plt.ylabel("Prediction Error (Actual - Predicted)")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12.5 XGBoost Feature Importance

Feature importance is examined to identify which variables contribute most to the electricity load predictions.

This helps interpret the machine learning model and identify the most useful temporal, historical, and energy-system variables.

In [ ]:
# XGBoost feature importance

import pandas as pd
import matplotlib.pyplot as plt

feature_importance = pd.Series(
    xgb_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("Top 15 Most Important Features:")
display(
    feature_importance.head(15).to_frame("importance")
)

# Plot top 15 features
plt.figure(figsize=(10, 7))

feature_importance.head(15).sort_values().plot(
    kind="barh"
)

plt.title("Top 15 XGBoost Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

## 12.6 Final Model Evaluation

The XGBoost forecasting model is compared with the 15-minute persistence baseline using MAE and RMSE.

Lower values indicate better forecasting performance.

In [ ]:
# Final comparison of baseline and XGBoost

comparison = pd.DataFrame({
    "Model": [
        "15-Minute Persistence Baseline",
        "XGBoost"
    ],
    "Validation MAE": [
        baseline_val_mae,
        xgb_val_mae
    ],
    "Validation RMSE": [
        baseline_val_rmse,
        xgb_val_rmse
    ],
    "Test MAE": [
        baseline_test_mae,
        xgb_test_mae
    ],
    "Test RMSE": [
        baseline_test_rmse,
        xgb_test_rmse
    ]
})

display(comparison.round(2))

## 13. Saving the Trained Forecasting Model

The trained XGBoost model is saved so that it can be reused by the interactive web application without retraining.

The feature names are also saved to ensure that future prediction inputs are supplied in the same structure as the training data.

In [ ]:
# Save the trained XGBoost model and feature information

import joblib

# Save model
joblib.dump(xgb_model, "xgboost_load_forecasting_model.pkl")

# Save the feature names used by the model
feature_names = list(X_train.columns)
joblib.dump(feature_names, "model_features.pkl")

print("Model saved successfully.")
print("Feature information saved successfully.")
print("Number of features:", len(feature_names))

# Dataset Overview & Data Quality

## Dataset Summary

This project uses a high-frequency German electricity dataset containing electricity load, generation, imports, and renewable-energy forecast variables.

| Property | Description |
|---|---|
| Dataset | German Electricity Consumption & Generation |
| Source | Kaggle |
| Time period | 2015 – 2026 |
| Original frequency | 15 minutes |
| Original observations | 395,375 |
| Target variable | `load_Actual Load` |
| Target | Actual German electricity load |
| Original variables | 87 measurement variables + timestamp |
| Final modelling variables | 60 |
| Final modelling observations | 379,962 |
| Forecasting task | 15-minute-ahead electricity load prediction |
| Machine learning model | XGBoost Regression |

## Main Feature Groups

The dataset contains information from several parts of the German electricity system:

- **Actual electricity load**
- **Electricity generation by energy source**
- **Electricity imports and exports**
- **Wind and solar generation**
- **Wind and solar forecasts**
- **Hydroelectric generation**
- **Fossil-fuel generation**
- **Nuclear generation**
- **Biomass and other renewable generation**
- **Calendar and time-based features**
- **Historical load features (lags)**
- **Rolling statistical features**

## Data Preparation

The raw dataset was preserved separately and all preprocessing was performed on working copies.

Key preprocessing steps included:

1. Timestamp parsing and chronological ordering.
2. Identification of missing timestamps and irregular time gaps.
3. Missing-value analysis across all variables.
4. Removal of variables with substantial missingness.
5. Interpolation of only small internal gaps in retained predictor variables.
6. Removal of rows where required modelling predictors were unavailable.
7. Removal of rows with missing target values.
8. Detection and correction of an extreme load outlier.
9. Creation of calendar, lag, and rolling statistical features.
10. Chronological train/validation/test splitting to prevent future information from entering the training data.

> **Important:** Long periods with missing timestamps were not artificially filled. This avoids creating synthetic electricity-load observations where the original dataset contains no measurements.

## Final Modelling Dataset

The final dataset contains:

- **379,962 observations**
- **60 total modelling variables**
- **59 input features**
- **1 target variable**
- **0 missing values**
- **0 duplicate timestamps**
- **Chronologically ordered observations**

The forecasting model predicts electricity load using recent historical load behaviour, calendar patterns, generation information, and other available system variables.

In [ ]:
# Continuous time segments using the cleaned dataset

# df_clean currently uses timestamp as its index
segment_data = df_clean[[target_column]].copy()

# Make timestamp an ordinary column for easier processing
segment_data = segment_data.reset_index()

# Make sure the timestamp column has the expected name
if "timestamp" not in segment_data.columns:
    segment_data = segment_data.rename(columns={segment_data.columns[0]: "timestamp"})

segment_data = segment_data.sort_values("timestamp").reset_index(drop=True)

# Calculate gap between consecutive observations
segment_data["time_gap"] = segment_data["timestamp"].diff()

# A new segment begins when the gap is greater than 15 minutes
segment_data["new_segment"] = (
    segment_data["time_gap"] > pd.Timedelta(minutes=15)
)

segment_data["segment_id"] = (
    segment_data["new_segment"].cumsum() + 1
)

# Create segment summary
segments = (
    segment_data
    .groupby("segment_id")
    .agg(
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        observations=("timestamp", "size")
    )
    .reset_index()
)

segments["duration"] = segments["end"] - segments["start"]

# Gap separating this segment from the previous segment
segments["gap_before"] = (
    segments["start"] - segments["end"].shift(1)
)

segments.loc[0, "gap_before"] = pd.Timedelta(0)

print("Continuous Time Segments — Cleaned Dataset")
print("=" * 70)

display(
    segments[
        [
            "segment_id",
            "start",
            "end",
            "duration",
            "observations",
            "gap_before"
        ]
    ]
)

print(f"\nNumber of continuous segments: {len(segments)}")

In [ ]:
# Plot cleaned electricity load with a different colour for each time segment

plt.figure(figsize=(18, 7))

for _, segment in segments.iterrows():

    segment_id = segment["segment_id"]

    segment_rows = segment_data[
        segment_data["segment_id"] == segment_id
    ]

    plt.plot(
        segment_rows["timestamp"],
        segment_rows[target_column],
        linewidth=0.7,
        label=f"Segment {int(segment_id)}"
    )

plt.title("German Electricity Load — Continuous Time Segments")
plt.xlabel("Time")
plt.ylabel("Actual Electricity Load")
plt.grid(True, alpha=0.3)

# Only display legend if it remains readable
if len(segments) <= 20:
    plt.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8
    )

plt.tight_layout()
plt.show()

In [ ]:
# Create a table of all missing timestamp periods

expected_timestamps = pd.date_range(
    start=segment_data["timestamp"].min(),
    end=segment_data["timestamp"].max(),
    freq="15min"
)

actual_timestamps = pd.DatetimeIndex(
    segment_data["timestamp"]
)

missing_timestamps = expected_timestamps.difference(
    actual_timestamps
)

# Group consecutive missing timestamps into periods
missing_df = pd.DataFrame({
    "timestamp": missing_timestamps
})

missing_df["gap_group"] = (
    missing_df["timestamp"].diff() != pd.Timedelta(minutes=15)
).cumsum()

missing_periods = (
    missing_df
    .groupby("gap_group")
    .agg(
        missing_start=("timestamp", "min"),
        missing_end=("timestamp", "max"),
        missing_records=("timestamp", "size")
    )
    .reset_index(drop=True)
)

missing_periods["missing_duration"] = (
    missing_periods["missing_end"]
    - missing_periods["missing_start"]
    + pd.Timedelta(minutes=15)
)

print("Missing Timestamp Periods")
print("=" * 80)

display(
    missing_periods[
        [
            "missing_start",
            "missing_end",
            "missing_records",
            "missing_duration"
        ]
    ]
)

print(
    f"Total missing 15-minute records: {len(missing_timestamps):,}"
)

In [ ]:
# Document the extreme load outlier correction

outlier_comparison = pd.DataFrame({
    "Timestamp": [
        "2026-06-25 16:30 UTC"
    ],
    "Initial Load": [
        4315702.57803
    ],
    "Corrected Load": [
        62423.10549
    ],
    "Reason": [
        "Extreme value inconsistent with surrounding electricity-load observations"
    ]
})

display(outlier_comparison)

### Outlier Correction

One extreme value in the actual electricity-load series was identified as inconsistent with the surrounding observations.

The recorded value was approximately **4.32 million**, while neighbouring observations were approximately **62,000 MW**. The value was therefore corrected to **62,423.11 MW** based on the surrounding load pattern.

This correction prevents the anomalous observation from disproportionately affecting statistical analysis and machine-learning model training.

# Data Quality Summary

The dataset contains several important characteristics that were considered during preprocessing.

### Timestamp Quality
- The original dataset begins at **2015-01-01 00:00:00+01:00**.
- UTC conversion changes the displayed timestamp of the first observation to **2014-12-31 23:00 UTC**, but this does **not** represent additional 2014 data.
- The dataset contains irregular timestamp gaps.
- Long missing periods were identified rather than artificially interpolated.

### Missing Data
- Missing values were analysed for every measurement variable.
- Variables with substantial missingness were excluded from the modelling dataset.
- Only small internal gaps in retained predictors were interpolated.
- Missing target observations were not fabricated.

### Outlier Quality
- One extreme load observation was identified and corrected.
- The corrected value is consistent with the surrounding load observations.

### Modelling Integrity
- Data was split chronologically into training, validation, and test sets.
- Random shuffling was avoided because this is a time-series forecasting problem.
- Historical lag and rolling features were created from previous observations.
- The final modelling dataset contains no missing values or duplicate timestamps.

These preprocessing decisions ensure that the forecasting model is evaluated using historical information without artificially creating unavailable observations.